PyTorch Tensors — Basics Exercise 01: Tensors, shapes, dtypes, devices, and operations. Run on Google Colab with GPU: Runtime → Change runtime type → T4 GPU

In [1]:
import torch

# Check PyTorch version and GPU availability
print("PyTorch version:", torch.__version__)
print()

# Check all available devices
print("CUDA available:", torch.cuda.is_available())
print("MPS available: ", torch.backends.mps.is_available())
print()

# Detect best available device
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

# If CUDA — print GPU details
if torch.cuda.is_available():
    print()
    print("GPU name:      ", torch.cuda.get_device_name(0))
    print("GPU memory:    ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

PyTorch version: 2.10.0+cpu

CUDA available: False
MPS available:  False

Using device: cpu


## 1. Creating Tensors

In [ ]:
# From data — dtype is inferred
t_int   = torch.tensor([1, 2, 3])
t_float = torch.tensor([1.0, 2.0, 3.0])

print("int tensor:  ", t_int,   " | dtype:", t_int.dtype)
print("float tensor:", t_float, " | dtype:", t_float.dtype)

In [ ]:
# Zeros, ones, random
print(torch.zeros(3, 4))          # 3×4 matrix of zeros
print(torch.ones(2, 3))           # 2×3 matrix of ones
print(torch.rand(2, 3))           # uniform random 0-1
print(torch.randn(2, 3))          # standard normal (mean=0, std=1)
print(torch.arange(0, 10, 2))     # [0, 2, 4, 6, 8] — like Python range()

## 2. Tensor Properties — shape, dtype, device

In [ ]:
t = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print("shape:  ", t.shape)          # torch.Size([2, 3])
print("rows:   ", t.shape[0])       # 2
print("cols:   ", t.shape[1])       # 3
print("dtype:  ", t.dtype)          # torch.float32
print("device: ", t.device)         # cpu
print("ndim:   ", t.ndim)           # 2 — number of dimensions
print("numel:  ", t.numel())        # 6 — total number of elements

## 3. dtype — casting between types

In [ ]:
t = torch.tensor([1, 2, 3])           # int64

print(t.dtype)                        # torch.int64
print(t.float().dtype)                # torch.float32  — most common in ML
print(t.half().dtype)                 # torch.float16  — used in mixed precision / LLM inference
print(t.long().dtype)                 # torch.int64    — used for class labels / indices
print(t.bool().dtype)                 # torch.bool     — used for masks

# Why float32 is default for ML:
# float16 → fast but loses precision (can cause NaN in training)
# float32 → good balance of speed and numerical stability
# float64 → too slow on GPU, unnecessary precision

## 4. Devices — moving tensors to GPU

In [ ]:
# Always write device-agnostic code — works on CPU, GPU, and Apple Silicon
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Using device:", device)

# Create tensor directly on device
t = torch.tensor([1.0, 2.0, 3.0], device=device)
print("tensor device:", t.device)

# Move existing tensor to device
t2 = torch.randn(3, 3)
t2 = t2.to(device)
print("t2 device:", t2.device)

# COMMON MISTAKE — two tensors must be on the same device
a = torch.tensor([1.0]).to(device)
b = torch.tensor([2.0])             # stays on CPU
try:
    c = a + b                       # will error if device != cpu
except RuntimeError as e:
    print("Error:", e)

## 5. Indexing and Slicing

In [ ]:
t = torch.tensor([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])

print(t[0])            # first row:           [1, 2, 3]
print(t[0, 1])         # row 0, col 1:        2
print(t[:, 1])         # all rows, col 1:     [2, 5, 8]
print(t[0:2, :])       # rows 0 and 1:        [[1,2,3],[4,5,6]]
print(t[-1])           # last row:            [7, 8, 9]

# Boolean mask — select elements matching condition
print(t[t > 4])        # [5, 6, 7, 8, 9]

## 6. Reshaping — reshape, view, squeeze, unsqueeze

In [ ]:
t = torch.arange(12)              # [0, 1, 2, ..., 11]  shape: [12]
print("original:      ", t.shape)

print("reshape(3,4):  ", t.reshape(3, 4).shape)     # [3, 4]
print("reshape(2,2,3):", t.reshape(2, 2, 3).shape)  # [2, 2, 3]
print("reshape(-1,4): ", t.reshape(-1, 4).shape)     # -1 = infer → [3, 4]

# view vs reshape
# view — no memory copy, must be contiguous (faster)
# reshape — may copy memory if needed (safer)
print("view(3,4):     ", t.view(3, 4).shape)

# unsqueeze — add a dimension of size 1
t2 = torch.tensor([1.0, 2.0, 3.0])   # shape: [3]
print("\nunsqueeze(0):  ", t2.unsqueeze(0).shape)  # [1, 3] — adds batch dim at front
print("unsqueeze(1):  ", t2.unsqueeze(1).shape)    # [3, 1]

# squeeze — removes all dims of size 1
t3 = torch.zeros(1, 3, 1)            # shape: [1, 3, 1]
print("squeeze():     ", t3.squeeze().shape)       # [3]

## 7. Math Operations

In [ ]:
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

# Element-wise
print("a + b:\n",  a + b)
print("a * b:\n",  a * b)    # element-wise — NOT matrix multiply

# Matrix multiply — the most important operation in ML
print("a @ b:\n",  a @ b)    # same as torch.matmul(a, b)

# Dot product (1D tensors only)
x = torch.tensor([1., 2., 3.])
y = torch.tensor([4., 5., 6.])
print("dot product:", torch.dot(x, y))   # (1×4)+(2×5)+(3×6) = 32

# Reductions
print("sum:    ", a.sum())
print("mean:   ", a.mean())
print("sum dim=0:", a.sum(dim=0))   # collapse rows → per column sum
print("sum dim=1:", a.sum(dim=1))   # collapse cols → per row sum

# Softmax — converts logits to probabilities
logits = torch.tensor([2.0, 1.0, 0.1])
probs = torch.softmax(logits, dim=0)
print("softmax:", probs)
print("sum of probs:", probs.sum())   # always 1.0